# Public Data Indicators for Ex-Ante Financial Control: Procedure BG-RRP-1.015 "School STEM Environment"

**Element 2 (E2)** of the Financial Controller qualification thesis on *"Ex-Ante Control for Legality and Fraud and Irregularity Indicators in the Award and Expenditure of Funds under Procedure BG-RRP-1.015 'School STEM Environment'"*.

**Methodological safeguard:** A statistical indicator in public procurement records is **not** a proven legal violation. Indicators identify measurable patterns before commitment that require documentary verification against internal financial control files (Element 3) and preventive checklist controls (Element 4). In accordance with public registry rules, legal entity and school names and EIK/Bulstat identifiers are preserved throughout tables and analyses, while sole-trader (`ЕТ`) and natural-person names are redacted.

**Author:** Valeriy Beluhov.

## 1. Purpose, Scope, Inputs, and Relational Database Preparation

This study measures ex-ante control indicators verifiable prior to commitment in public data for procedure `BG-RRP-1.015` (National Recovery and Resilience Plan) under Art. 13(3) items 1 and 3 of the Financial Management and Control in the Public Sector Act (FMCPSA), Ministry of Finance Guidelines `ZMF-184/06.03.2020`, and Arts. 17, 20, and 21 of the Public Procurement Act (PPA).

**Working Hypothesis (H2):** Pre-specified decision rule — confirmed if at least two of the three contract-level indicators (`predated`, `split_30d`, `regime_mismatch`) exhibit a lower 95% Wilson confidence bound exceeding **5%** (`0.05`) across participating schools with contracts (`N = 2,222`).

**Primary Inputs (read-only):**
- `openData.xml` and `openData.xsd` (`EXTRACTION_DATE = "2026-09-10"`): ISUN 2020 open data export hashed in `data/MANIFEST.csv`;
- Upstream Element 1 (`adfi-violations-classification@c9bd90d`): `adfi_reports_meta.csv`, `adfi_classification.csv`, `adfi_classification_verbatim.csv`, `classifier_classes.csv`.


In [1]:
from __future__ import annotations

import csv
import math
import re
from difflib import SequenceMatcher
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import networkx as nx
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import sparse, stats
from statsmodels.discrete.discrete_model import Poisson

EXTRACTION_DATE = "2026-09-10"
PROCEDURE_PREFIX = "BG-RRP-1.015"
EUR_BGN = 1.95583
VAT_FACTOR = 1.20
RATIO_TOLERANCE = 0.03
REGIME_1_CUTOFF = "2023-12-31"
EURO_FROM = "2026-01-01"
RNG_STATE = 2026
WILSON_Z = 1.95996

DATA_DIR = Path("data")
DERIVED_DIR = DATA_DIR / "derived"
FIG_DIR = Path("fig")
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.sans-serif": ["DejaVu Sans"],
    "axes.edgecolor": "#444444",
    "axes.linewidth": 0.8,
})

conclusions_log = []


def record_conclusion(part, question, measure, value, interval_low="", interval_high="", interval_method="none", decision="none"):
    """Append one verified quantitative metric row for conclusions_summary.csv."""
    conclusions_log.append({
        "part": str(part),
        "question": str(question),
        "measure": str(measure),
        "value": str(value),
        "interval_low": "" if interval_low == "" else str(interval_low),
        "interval_high": "" if interval_high == "" else str(interval_high),
        "interval_method": str(interval_method),
        "decision": str(decision),
    })


def flush_conclusions():
    """Persist conclusions_summary.csv using Element 1 semicolon-delimited format."""
    df_out = pd.DataFrame(
        conclusions_log,
        columns=["part", "question", "measure", "value", "interval_low", "interval_high", "interval_method", "decision"],
    )
    df_out.to_csv(DERIVED_DIR / "conclusions_summary.csv", sep=";", index=False, encoding="utf-8", lineterminator="\n")
    return df_out


def wilson_ci(k: int, n: int, z: float = WILSON_Z):
    """Return (proportion, lower_95, upper_95) using the Wilson score interval."""
    if n <= 0:
        return 0.0, 0.0, 0.0
    p = k / n
    denom = 1.0 + z * z / n
    center = (p + z * z / (2.0 * n)) / denom
    half = z * math.sqrt(p * (1.0 - p) / n + z * z / (4.0 * n * n)) / denom
    return p, max(0.0, center - half), min(1.0, center + half)


def bh_fdr(p_values):
    """Compute Benjamini-Hochberg adjusted q-values."""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p, kind="stable")
    scaled = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.empty_like(p)
    q[order] = np.minimum(np.minimum.accumulate(scaled[::-1])[::-1], 1.0)
    return q


In [2]:
isun_projects = pd.read_csv(DATA_DIR / "isun_projects.csv", dtype=str, keep_default_na=False)
for col in ("total_value", "actually_paid_amount"):
    isun_projects[col] = pd.to_numeric(isun_projects[col])

isun_contracts = pd.read_csv(DATA_DIR / "isun_contracts.csv", dtype=str, keep_default_na=False)
for col in ("amount_raw", "subcontractors_count"):
    isun_contracts[col] = pd.to_numeric(isun_contracts[col])
isun_contracts["subcontractors_count"] = isun_contracts["subcontractors_count"].astype(int)

isun_entities = pd.read_csv(DATA_DIR / "isun_entities.csv", dtype=str, keep_default_na=False)
schools_register = pd.read_csv(DATA_DIR / "schools_register.csv", dtype=str, keep_default_na=False)
for col in schools_register.columns[9:]:
    schools_register[col] = pd.to_numeric(schools_register[col])

adfi_meta = pd.read_csv(DATA_DIR / "adfi_reports_meta.csv", sep=";", dtype=str, keep_default_na=False)
adfi_cls = pd.read_csv(DATA_DIR / "adfi_classification.csv", sep=",", dtype=str, keep_default_na=False)
adfi_verb = pd.read_csv(DATA_DIR / "adfi_classification_verbatim.csv", sep=";", dtype=str, keep_default_na=False)
classifier_classes = pd.read_csv(DATA_DIR / "classifier_classes.csv", sep=";", dtype=str, keep_default_na=False)

contracts = pd.read_csv(DERIVED_DIR / "contracts.csv", dtype=str, keep_default_na=False)
for col in ("contract_row", "delta_t_days", "amount_raw", "amount_eur_incl_vat", "amount_eur_excl_vat", "amount_bgn_excl_vat", "paid_alloc_eur_incl_vat", "subcontractors_count"):
    contracts[col] = pd.to_numeric(contracts[col])

schools = pd.read_csv(DERIVED_DIR / "schools.csv", dtype=str, keep_default_na=False)
for col in ("projects_count", "contracts_count", "register_budget_bgn_incl_vat", "contracted_eur_incl_vat", "contracted_eur_excl_vat", "paid_eur_incl_vat"):
    schools[col] = pd.to_numeric(schools[col])
for col in ("flag_predated", "flag_split_30d", "flag_regime_mismatch", "flag_level1", "flag_bunching_50k"):
    schools[col] = schools[col] == "True"

project_classes = pd.read_csv(DERIVED_DIR / "project_classes.csv", dtype=str, keep_default_na=False)
for col in ("total_value", "actually_paid_amount", "contract_sum_raw", "register_budget_bgn_incl_vat", "register_budget_eur_incl_vat", "paid_eur_incl_vat"):
    project_classes[col] = pd.to_numeric(project_classes[col])

normalized_totals = pd.read_csv(DERIVED_DIR / "normalized_totals.csv")
flags_tbl = pd.read_csv(DERIVED_DIR / "forensic_risk_flags.csv", dtype=str, keep_default_na=False)
for col in ("contract_row", "amount_raw", "amount_eur_excl_vat", "amount_bgn_excl_vat", "threshold_bgn_excl_vat"):
    flags_tbl[col] = pd.to_numeric(flags_tbl[col])
for col in ("flag_predated", "flag_split_30d", "flag_same_day_sub", "flag_regime_mismatch", "flag_level1", "flag_bunching_30k", "flag_bunching_50k", "flag_bunching_80k"):
    flags_tbl[col] = flags_tbl[col] == "True"

assert len(isun_projects) == 2228
assert len(isun_contracts) == 11128
assert len(contracts) == 11128
assert len(schools) == 2222
assert "vendor_role" not in contracts.columns
assert "name" in isun_entities.columns and "vendor_name" in contracts.columns and "school_name" in schools.columns
assert not any("zdoi" in c.lower() for c in schools.columns)

print(f"Loaded {len(isun_projects):,} projects, {len(isun_contracts):,} contracts, {len(isun_entities):,} entities, {len(schools_register):,} register institutions.")
print(f"Loaded {len(adfi_meta):,} ADFI reports ({len(adfi_cls):,} classification rows) and {len(schools):,} STEM schools with contracts.")


Loaded 2,228 projects, 11,128 contracts, 3,973 entities, 2,271 register institutions.
Loaded 560 ADFI reports (1,824 classification rows) and 2,222 STEM schools with contracts.


## 2. Currency and VAT Normalization and Verification of Structural Invariants

ISUN 2020 mixes legacy BGN entries with post-euro entries, with or without 20% VAT. The project is the unit of normalization. The sum of its contracts is divided by the project's budget ceiling in the `BG-RRP-1.015` register (BGN incl. VAT), and the ratio is matched, within a 3% tolerance, to one of four canonical values: `1.0`, `1/1.2`, `1/1.95583`, `1/(1.95583*1.2)`. The match gives the currency and the VAT basis, and every contract of the project inherits both.

Three rules complete the procedure. Contracts signed on or after 2026-01-01 are taken as EUR. Where the ceiling ratio is inconclusive, the currency is inferred from the ratio of paid to contracted amounts, and EUR is assumed when that ratio is inconclusive too. Where the VAT basis stays undetermined, the dominant basis (`incl_vat`) is assumed.

Both bases are needed for financial control:
- **`amount_eur_incl_vat`**: the basis for internal budgetary ceiling checks prior to commitment under Art. 13(3), item 3 FMCPSA;
- **`amount_bgn_excl_vat` / `amount_eur_excl_vat`**: the basis for the procurement thresholds of Art. 20 PPA (`30,000`, `50,000`, `80,000 BGN excl. VAT`). The 20% VAT difference inside `[41,667; 60,000] BGN` decides whether a contract falls below or above the `50,000 BGN` threshold (Class `1.02` in Element 1).

The next cell asserts the structural invariants (counts of projects, schools, contracts, suppliers and register matches, and the agreement of contract sums with the normalized totals) and writes the reconciliation to `data/derived/reconciliation_summary.csv`.

In [3]:
n_projects = len(isun_projects)
n_beneficiaries = project_classes["school_bulstat"].nunique()
n_projects_with_contract = contracts["project_id"].nunique()
n_schools_with_contract = contracts["school_bulstat"].nunique()
n_projects_without_contract = n_projects - n_projects_with_contract
n_contracts = len(contracts)
n_unique_cids = contracts["contract_id"].nunique()
n_duplicate_cids = int(contracts["contract_id"].duplicated().sum())
non_empty_vendors = contracts.loc[contracts["vendor_id"] != "", "vendor_id"]
n_vendors_id = non_empty_vendors.nunique()
n_vendors_eik = contracts.loc[contracts["vendor_eik"] != "", "vendor_eik"].nunique()
n_contracts_no_eik = int((contracts["vendor_eik"] == "").sum())
n_contracts_no_vid = int((contracts["vendor_id"] == "").sum())

total_incl_vat = float(normalized_totals.loc[0, "total_contracted_eur_incl_vat"])
total_excl_vat = float(normalized_totals.loc[0, "total_contracted_eur_excl_vat"])
total_vat_national = round(total_incl_vat - total_excl_vat, 2)
total_paid = float(normalized_totals.loc[0, "total_disbursed_eur_incl_vat"])
total_unpaid = float(normalized_totals.loc[0, "total_unpaid_eur_incl_vat"])
n_default_curr = int(normalized_totals.loc[0, "contracts_currency_by_default"])
n_assumed_vat = int(normalized_totals.loc[0, "contracts_vat_assumed"])
vat_assumed_raw = float(normalized_totals.loc[0, "vat_assumed_amount_raw"])

known_vat_proj = project_classes[project_classes["contract_vat_basis"] != "undetermined"]
incl_share_known = float((known_vat_proj["contract_vat_basis"] == "incl_vat").mean())

assert n_projects == 2228
assert n_beneficiaries == 2227
assert n_projects_with_contract == 2223 and n_projects_without_contract == 5
assert n_schools_with_contract == 2222
assert n_contracts == 11128 and contracts["contract_row"].is_unique
assert n_vendors_id == 1754 and n_vendors_eik == 1732 and n_contracts_no_eik == 46
assert (project_classes["register_match"] == "bulstat").sum() == 2075
assert (project_classes["register_match"] == "name_settlement").sum() == 153
assert abs(contracts["amount_eur_incl_vat"].sum() - total_incl_vat) < 0.02
assert abs(contracts["amount_eur_excl_vat"].sum() - total_excl_vat) < 0.02

reconcil_rows = [
    ("Projects in procedure BG-RRP-1.015", n_projects, "ISUN openData.xml"),
    ("Unique beneficiary schools (register matched)", n_beneficiaries, "One school holds two projects (100% matched)"),
    ("Projects with at least one contract", n_projects_with_contract, "Across 2,222 schools; 5 projects have 0 contracts"),
    ("Contract rows (key: contract_row)", n_contracts, f"{n_duplicate_cids} duplicate contract_id strings"),
    ("Identified suppliers (vendor_id / EIK)", f"{n_vendors_id} / {n_vendors_eik}", f"{n_contracts_no_eik} contracts lack EIK; {n_contracts_no_vid} lacks vendor_id"),
    ("Matched by Bulstat / name+settlement", "2075 / 153", "100% matched against raw MoES register"),
    ("Contracted value incl. VAT (EUR)", f"{total_incl_vat:,.2f}", "Control basis under Art. 13(3)(3) FMCPSA"),
    ("Contracted value excl. VAT (EUR)", f"{total_excl_vat:,.2f}", "EU RRF grant basis; Art. 20 PPA threshold basis"),
    ("National VAT co-financing (EUR)", f"{total_vat_national:,.2f}", "20% VAT component"),
    ("Disbursed payments incl. VAT (EUR)", f"{total_paid:,.2f}", f"{total_paid / total_incl_vat:.2%} of contracted total"),
    ("Unpaid remainder incl. VAT (EUR)", f"{total_unpaid:,.2f}", f"{total_unpaid / total_incl_vat:.2%} open to ex-ante expenditure control"),
    ("Contracts with assumed incl_vat regime", n_assumed_vat, f"Raw sum {vat_assumed_raw:,.2f}; {incl_share_known:.1%} of classified projects are incl_vat"),
]
reconcil_df = pd.DataFrame(reconcil_rows, columns=["invariant", "computed_value", "notes"])
reconcil_df.to_csv(DERIVED_DIR / "reconciliation_summary.csv", index=False, encoding="utf-8", lineterminator="\n")
display(reconcil_df.style.hide(axis="index"))
display(contracts["currency_basis"].value_counts().rename_axis("currency_basis").to_frame("contracts"))

record_conclusion("Section 2", "reconciliation", "projects in BG-RRP-1.015", n_projects)
record_conclusion("Section 2", "reconciliation", "unique beneficiary schools", n_beneficiaries)
record_conclusion("Section 2", "reconciliation", "projects with contract", n_projects_with_contract)
record_conclusion("Section 2", "reconciliation", "schools with contract", n_schools_with_contract)
record_conclusion("Section 2", "reconciliation", "projects without contract", n_projects_without_contract)
record_conclusion("Section 2", "reconciliation", "contracts count", n_contracts)
record_conclusion("Section 2", "reconciliation", "unique contract_id", n_unique_cids)
record_conclusion("Section 2", "reconciliation", "distinct non-empty vendor_id", n_vendors_id)
record_conclusion("Section 2", "reconciliation", "distinct vendor_eik", n_vendors_eik)
record_conclusion("Section 2", "reconciliation", "contracts without vendor_eik", n_contracts_no_eik)
record_conclusion("Section 2", "reconciliation", "total contracted EUR incl VAT", f"{total_incl_vat:.2f}")
record_conclusion("Section 2", "reconciliation", "total contracted EUR excl VAT", f"{total_excl_vat:.2f}")
record_conclusion("Section 2", "reconciliation", "national VAT co-financing EUR", f"{total_vat_national:.2f}")
record_conclusion("Section 2", "reconciliation", "total disbursed EUR incl VAT", f"{total_paid:.2f}")
record_conclusion("Section 2", "reconciliation", "total unpaid remainder EUR incl VAT", f"{total_unpaid:.2f}")
record_conclusion("Section 2", "reconciliation", "unpaid share of contracted value", f"{total_unpaid / total_incl_vat:.6f}")
record_conclusion("Section 2", "reconciliation", "contracts with default currency", n_default_curr)
record_conclusion("Section 2", "reconciliation", "contracts with dominant VAT assumption", n_assumed_vat)
record_conclusion("Section 2", "reconciliation", "assumed VAT raw amount", f"{vat_assumed_raw:.2f}")
record_conclusion("Section 2", "reconciliation", "share of classified projects with incl VAT", f"{incl_share_known:.6f}")
flush_conclusions()
print(f"Currency basis breakdown: {contracts['currency_basis'].value_counts().to_dict()}")


invariant,computed_value,notes
Projects in procedure BG-RRP-1.015,2228,ISUN openData.xml
Unique beneficiary schools (register matched),2227,One school holds two projects (100% matched)
Projects with at least one contract,2223,"Across 2,222 schools; 5 projects have 0 contracts"
Contract rows (key: contract_row),11128,454 duplicate contract_id strings
Identified suppliers (vendor_id / EIK),1754 / 1732,46 contracts lack EIK; 1 lacks vendor_id
Matched by Bulstat / name+settlement,2075 / 153,100% matched against raw MoES register
Contracted value incl. VAT (EUR),"288,593,366.12",Control basis under Art. 13(3)(3) FMCPSA
Contracted value excl. VAT (EUR),"240,494,471.77",EU RRF grant basis; Art. 20 PPA threshold basis
National VAT co-financing (EUR),"48,098,894.35",20% VAT component
Disbursed payments incl. VAT (EUR),"260,070,052.15",90.12% of contracted total


,contracts
currency_basis,
ceiling,5166
default,2221
date,2203
payment,1538


Currency basis breakdown: {'ceiling': 5166, 'default': 2221, 'date': 2203, 'payment': 1538}


## 3. Normative Concept Mapping and Bridge to Element 1 (ADFI Classifier)

Every public-data indicator is operationalized strictly as a control check before commitment under Art. 13(3) items 1 and 3 FMCPSA and linked directly to its corresponding violation classes in Element 1 (`adfi-violations-classification@c9bd90d`). Following the author's statutory specification (Decision 6 and Decision 7), because ISUN records uniform text without distinguishing supplies/services from construction works, all contract-level and bunching evaluations apply the statutory **goods and services thresholds** at the signature date (`30,000 BGN excl. VAT` up to `31.12.2023`; `50,000 BGN excl. VAT` from `01.01.2024`, SG No. 88/20.10.2023). Works limits (`50,000 / 80,000 BGN excl. VAT`) are mapped normatively and shown strictly as visual reference lines in Figure `F2_1`.


In [4]:
cls_summary_e1 = classifier_classes.set_index("class_index")
total_e1_rows = int(pd.to_numeric(classifier_classes["rows_total"]).sum())
total_e1_viol = int(pd.to_numeric(classifier_classes["rows_violation"]).sum())
total_e1_nv = int(pd.to_numeric(classifier_classes["rows_no_violation"]).sum())
sec_24_viol = int(pd.to_numeric(classifier_classes.loc[classifier_classes["section_no"].isin(["2", "3", "4"]), "rows_violation"]).sum())
assert total_e1_rows == 1824 and total_e1_viol == 1203 and total_e1_nv == 621

sig_dt = pd.to_datetime(contracts["signature_date"])
contracts["regime"] = np.where(sig_dt <= pd.Timestamp(REGIME_1_CUTOFF), 1, 2)
n_reg1 = int((contracts["regime"] == 1).sum())
n_reg2 = int((contracts["regime"] == 2).sum())
assert n_reg1 == 73 and n_reg2 == 11055

def e1_counts(idx_list):
    parts = []
    for ci in idx_list:
        r = cls_summary_e1.loc[ci]
        parts.append(f"{ci}: {r['rows_violation']} viol / {r['rows_no_violation']} no-viol ({r['reports']} reports)")
    return "; ".join(parts)

concepts_rows = [
    {
        "indicator": "predated",
        "bg_term": "Договор преди старта на проекта",
        "control_point": "Ex-ante check before commitment: time eligibility & project start chronology",
        "legal_basis": "Art. 13(3)(3) FMCPSA; ZMF-184/2020 Section IV",
        "threshold_bgn_excl_vat": "N/A (delta_t_days < 0)",
        "e1_classes": "10.01; 10.02; 7.03",
        "e1_findings_summary": e1_counts(["10.01", "10.02", "7.03"]),
    },
    {
        "indicator": "split_30d",
        "bg_term": "Разделяне на поръчки в 30-дневен прозорец",
        "control_point": "Ex-ante check before commitment: anti-splitting aggregation across 30 days (incl. same day)",
        "legal_basis": "Art. 21(14)-(15) & Art. 20(4) PPA; Art. 13(3)(3) FMCPSA",
        "threshold_bgn_excl_vat": "Regime 1 (<=2023-12-31): 30,000 BGN (73 contracts); Regime 2 (>=2024-01-01): 50,000 BGN (11,055 contracts)",
        "e1_classes": "1.01",
        "e1_findings_summary": e1_counts(["1.01"]),
    },
    {
        "indicator": "regime_mismatch",
        "bg_term": "Несъответствие с приложимия праг по Режим 1",
        "control_point": "Ex-ante check before commitment: statutory threshold at signature date",
        "legal_basis": "Art. 17(1) & Art. 20(3)-(4) PPA",
        "threshold_bgn_excl_vat": "30,000 <= value < 50,000 BGN (Regime 1); >=50,000 tracked alongside",
        "e1_classes": "1.02",
        "e1_findings_summary": e1_counts(["1.02"]),
    },
    {
        "indicator": "bunching_50k",
        "bg_term": "Ценово скупчване под прага от 50 000 лв.",
        "control_point": "Ex-ante check before commitment: market-tested estimated value",
        "legal_basis": "Art. 21(1)-(2) & Art. 20(4) PPA",
        "threshold_bgn_excl_vat": "[47,000.00; 50,000.00) BGN excl. VAT under Regime 2 (works 50k/80k shown in F2_1 only)",
        "e1_classes": "1.01; 1.06",
        "e1_findings_summary": e1_counts(["1.01", "1.06"]),
    },
]
concepts_df = pd.DataFrame(concepts_rows)
concepts_df.to_csv(DERIVED_DIR / "concepts_mapping.csv", index=False, encoding="utf-8", lineterminator="\n")
display(concepts_df.style.hide(axis="index"))

record_conclusion("Section 3", "e1_bridge", "E1 reports in register", int(len(adfi_meta)))
record_conclusion("Section 3", "e1_bridge", "E1 classified rows (violation / no violation)", f"{total_e1_rows} ({total_e1_viol} / {total_e1_nv})")
record_conclusion("Section 3", "e1_bridge", "E1 documentation sections 2-4 violation share", f"{sec_24_viol / total_e1_viol:.6f}")
record_conclusion("Section 3", "e1_bridge", "contracts under Regime 1 (up to 31.12.2023)", n_reg1)
record_conclusion("Section 3", "e1_bridge", "contracts under Regime 2 (from 01.01.2024)", n_reg2)
flush_conclusions()
print(f"E1 bridge verified: Sections 2-4 hold {sec_24_viol}/{total_e1_viol} ({sec_24_viol/total_e1_viol:.1%}) violations; Regime 1={n_reg1}, Regime 2={n_reg2}.")


indicator,bg_term,control_point,legal_basis,threshold_bgn_excl_vat,e1_classes,e1_findings_summary
predated,Договор преди старта на проекта,Ex-ante check before commitment: time eligibility & project start chronology,Art. 13(3)(3) FMCPSA; ZMF-184/2020 Section IV,N/A (delta_t_days < 0),10.01; 10.02; 7.03,10.01: 22 viol / 5 no-viol (11 reports); 10.02: 20 viol / 5 no-viol (10 reports); 7.03: 2 viol / 0 no-viol (2 reports)
split_30d,Разделяне на поръчки в 30-дневен прозорец,Ex-ante check before commitment: anti-splitting aggregation across 30 days (incl. same day),Art. 21(14)-(15) & Art. 20(4) PPA; Art. 13(3)(3) FMCPSA,"Regime 1 (<=2023-12-31): 30,000 BGN (73 contracts); Regime 2 (>=2024-01-01): 50,000 BGN (11,055 contracts)",1.01,1.01: 10 viol / 4 no-viol (14 reports)
regime_mismatch,Несъответствие с приложимия праг по Режим 1,Ex-ante check before commitment: statutory threshold at signature date,Art. 17(1) & Art. 20(3)-(4) PPA,"30,000 <= value < 50,000 BGN (Regime 1); >=50,000 tracked alongside",1.02,1.02: 12 viol / 4 no-viol (12 reports)
bunching_50k,Ценово скупчване под прага от 50 000 лв.,Ex-ante check before commitment: market-tested estimated value,Art. 21(1)-(2) & Art. 20(4) PPA,"[47,000.00; 50,000.00) BGN excl. VAT under Regime 2 (works 50k/80k shown in F2_1 only)",1.01; 1.06,1.01: 10 viol / 4 no-viol (14 reports); 1.06: 0 viol / 34 no-viol (31 reports)


E1 bridge verified: Sections 2-4 hold 903/1203 (75.1%) violations; Regime 1=73, Regime 2=11055.


## 4. Contract-Level Ex-Ante Indicators (`predated`, `split_30d`, `regime_mismatch`)

We measure the three primary contract-level indicators verifiable before signing:
1. **`predated`** (`delta_t_days < 0`): contracts signed before official ISUN project start;
2. **`split_30d`** (Decision 6): sub-threshold contracts (`0 < amount_bgn_excl_vat < threshold`) of the same school and supplier within 30 calendar days (`|d_i - d_j| <= 30`) whose sub-threshold sum reaches or exceeds the applicable goods/services limit (`30,000 BGN` in Regime 1, `50,000 BGN` in Regime 2); strictly encompasses same-day sub-threshold splits (`638` contracts across `241` schools). For methodological comparison with `stem-collusion-npvu`, the broader `SplittingDetector` specification (which also sums above-threshold contracts alongside small contracts) is computed and reported alongside;
3. **`regime_mismatch`**: Regime 1 contracts (`<= 31.12.2023`) in `[30,000; 50,000)` BGN excl. VAT, plus Regime 1 contracts `>= 50,000` BGN excl. VAT reported transparently.

We also check sensitivity to the 8 anomalous signature dates outside `[2023-01-01; 2026-09-10]`.


In [5]:
c_eval = contracts.merge(
    flags_tbl[["contract_row", "flag_predated", "flag_split_30d", "flag_same_day_sub", "flag_regime_mismatch", "threshold_bgn_excl_vat"]],
    on="contract_row",
    validate="one_to_one",
)
c_eval["flag_reg1_ge_30k"] = (c_eval["regime"] == 1) & (c_eval["amount_bgn_excl_vat"] >= 30000.0)
N_SCHOOLS = 2222

def summarize_indicator(mask: pd.Series, ind_key: str, label: str, in_h2: bool, sets_count: str = "", notes: str = ""):
    sub = c_eval[mask]
    n_c = len(sub)
    n_p = sub["project_id"].nunique()
    n_s = sub["school_bulstat"].nunique()
    val_incl = float(sub["amount_eur_incl_vat"].sum())
    val_excl = float(sub["amount_eur_excl_vat"].sum())
    paid = float(sub["paid_alloc_eur_incl_vat"].sum())
    unpaid = val_incl - paid
    sh_c = n_c / len(c_eval)
    p_s, lo_s, hi_s = wilson_ci(n_s, N_SCHOOLS)
    over_5 = bool(lo_s > 0.05) if in_h2 else ""
    return {
        "indicator": ind_key,
        "label": label,
        "n_contracts": n_c,
        "n_projects": n_p,
        "n_sets_windows": sets_count,
        "n_schools": n_s,
        "value_eur_incl_vat": round(val_incl, 2),
        "value_eur_excl_vat": round(val_excl, 2),
        "paid_eur_incl_vat": round(paid, 2),
        "unpaid_eur_incl_vat": round(unpaid, 2),
        "share_contracts": round(sh_c, 6),
        "share_schools": round(p_s, 6),
        "ci95_low": round(lo_s, 6),
        "ci95_high": round(hi_s, 6),
        "in_H2_rule": in_h2,
        "ci95_lower_over_5pct": over_5,
        "notes": notes,
    }

vals_b = c_eval["amount_bgn_excl_vat"].to_numpy()
thrs_b = c_eval["threshold_bgn_excl_vat"].to_numpy()
days_b = (pd.to_datetime(c_eval["signature_date"]) - pd.Timestamp("1970-01-01")).dt.days.to_numpy()
sch_c = pd.factorize(c_eval["school_bulstat"])[0]
ven_c = pd.factorize(c_eval["vendor_id"])[0]
grp_c = pd.factorize(sch_c.astype(np.int64) * (ven_c.max() + 1) + ven_c)[0]

sub_only_idx = np.flatnonzero((vals_b > 0) & (vals_b < thrs_b))
ord_sub = sub_only_idx[np.lexsort((days_b[sub_only_idx], grp_c[sub_only_idx]))]
n_split_30d_clusters = 0
for sl in np.split(ord_sub, np.flatnonzero(np.diff(grp_c[ord_sub])) + 1):
    if len(sl) < 2:
        continue
    d_sl = days_b[sl]
    hit_sl = c_eval["flag_split_30d"].to_numpy()[sl]
    if hit_sl.any():
        n_split_30d_clusters += 1 + int((np.diff(d_sl[hit_sl]) > 30).sum())

assert (c_eval.loc[c_eval["flag_same_day_sub"], "flag_split_30d"]).all(), "Same-day split must be strict subset of split_30d"

pos_idx = np.flatnonzero(vals_b > 0)
ord_pos = np.lexsort((days_b[pos_idx], grp_c[pos_idx]))
key_pos = grp_c[pos_idx][ord_pos].astype(np.int64) * 10**6 + days_b[pos_idx][ord_pos]
cum_pos = np.concatenate([[0.0], np.cumsum(vals_b[pos_idx][ord_pos])])
end_30 = np.searchsorted(key_pos, key_pos, side="right")
beg_30 = np.searchsorted(key_pos, key_pos - 29, side="left")
comp_det30 = np.zeros(len(c_eval), dtype=bool)
comp_det30[pos_idx[ord_pos[((end_30 - beg_30) >= 2) & (vals_b[pos_idx][ord_pos] < thrs_b[pos_idx][ord_pos]) & ((cum_pos[end_30] - cum_pos[beg_30]) >= thrs_b[pos_idx][ord_pos])]]] = True

t21_rows = [
    summarize_indicator(c_eval["flag_predated"], "predated", "Contract signed before project start", True, "", "delta_t_days < 0"),
    summarize_indicator(c_eval["flag_split_30d"], "split_30d", "30-day sub-threshold order splitting", True, str(n_split_30d_clusters), f"30-day sub-threshold windows >= T; includes {int(c_eval['flag_same_day_sub'].sum())} same-day contracts in {c_eval.loc[c_eval['flag_same_day_sub'], 'school_bulstat'].nunique()} schools"),
    summarize_indicator(c_eval["flag_regime_mismatch"], "regime_mismatch", "Regime 1 award in [30k, 50k) BGN excl. VAT", True, "", "Statutory threshold in force <= 2023-12-31"),
    summarize_indicator(c_eval["flag_reg1_ge_30k"], "regime_mismatch_incl_50k", "Regime 1 awards >= 30,000 BGN excl. VAT (total)", False, "", "Includes 5 Regime 1 contracts >= 50,000 BGN excl. VAT"),
]
t21_df = pd.DataFrame(t21_rows)
t21_df.to_csv(DERIVED_DIR / "T2_1_indicators.csv", index=False, encoding="utf-8", lineterminator="\n")
display(t21_df.style.hide(axis="index"))

sig_ts = pd.to_datetime(c_eval["signature_date"])
anom_pre = sig_ts < pd.Timestamp("2023-01-01")
anom_post = sig_ts > pd.Timestamp(EXTRACTION_DATE)
anom_all = anom_pre | anom_post
c_clean = c_eval[~anom_all]
print(f"Date anomalies: {int(anom_all.sum())} ({int(anom_pre.sum())} pre-2023, {int(anom_post.sum())} post-{EXTRACTION_DATE}).")
print(f"Excluding anomalies: predated = {int(c_clean['flag_predated'].sum())} contracts in {c_clean.loc[c_clean['flag_predated'], 'school_bulstat'].nunique()} schools; split_30d = {int(c_clean['flag_split_30d'].sum())} contracts in {c_clean.loc[c_clean['flag_split_30d'], 'school_bulstat'].nunique()} schools.")
print(f"Comparative SplittingDetector (incl. above-T co-contracts in 30d trailing window): {int(comp_det30.sum())} contracts in {c_eval.loc[comp_det30, 'school_bulstat'].nunique()} schools.")

for col, nm in (("flag_predated", "predated"), ("flag_split_30d", "split_30d"), ("flag_regime_mismatch", "regime_mismatch")):
    top_v = c_eval[c_eval[col]].groupby(["vendor_eik", "vendor_name"]).agg(
        contracts=("contract_row", "count"),
        schools=("school_bulstat", "nunique"),
        value_eur_incl_vat=("amount_eur_incl_vat", "sum"),
    ).sort_values(["contracts", "value_eur_incl_vat"], ascending=False).head(5).reset_index()
    print(f"\nTop suppliers for {nm}:")
    display(top_v.style.hide(axis="index").format({"value_eur_incl_vat": "{:,.2f}"}))

for row in t21_rows:
    k = row["indicator"]
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: contracts", row["n_contracts"])
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: projects", row["n_projects"])
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: schools", row["n_schools"])
    if row["n_sets_windows"]:
        record_conclusion("Section 4", f"indicator_{k}", f"{k}: 30-day clusters", row["n_sets_windows"])
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: value EUR incl VAT", f"{row['value_eur_incl_vat']:.2f}")
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: value EUR excl VAT", f"{row['value_eur_excl_vat']:.2f}")
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: paid EUR incl VAT", f"{row['paid_eur_incl_vat']:.2f}")
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: unpaid EUR incl VAT", f"{row['unpaid_eur_incl_vat']:.2f}")
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: share of contracts", f"{row['share_contracts']:.6f}")
    record_conclusion("Section 4", f"indicator_{k}", f"{k}: share of schools", f"{row['share_schools']:.6f}", f"{row['ci95_low']:.6f}", f"{row['ci95_high']:.6f}", "Wilson", str(row["ci95_lower_over_5pct"]) if row["in_H2_rule"] else "none")

record_conclusion("Section 4", "indicator_split_30d", "same_day_sub_split: contracts (subset of split_30d)", int(c_eval["flag_same_day_sub"].sum()))
record_conclusion("Section 4", "indicator_split_30d", "same_day_sub_split: schools (subset of split_30d)", int(c_eval.loc[c_eval["flag_same_day_sub"], "school_bulstat"].nunique()))
record_conclusion("Section 4", "indicator_split_30d", "comparative SplittingDetector 30d: contracts", int(comp_det30.sum()))
record_conclusion("Section 4", "indicator_split_30d", "comparative SplittingDetector 30d: schools", int(c_eval.loc[comp_det30, "school_bulstat"].nunique()))
record_conclusion("Section 4", "anomalies", "anomalous dates total (pre-2023 / post-extraction)", f"{int(anom_all.sum())} ({int(anom_pre.sum())} / {int(anom_post.sum())})")
record_conclusion("Section 4", "anomalies", "predated schools excluding date anomalies", int(c_clean.loc[c_clean["flag_predated"], "school_bulstat"].nunique()))
flush_conclusions()


indicator,label,n_contracts,n_projects,n_sets_windows,n_schools,value_eur_incl_vat,value_eur_excl_vat,paid_eur_incl_vat,unpaid_eur_incl_vat,share_contracts,share_schools,ci95_low,ci95_high,in_H2_rule,ci95_lower_over_5pct,notes
predated,Contract signed before project start,654,380,,380,11171740.450000,9309783.710000,10536305.150000,635435.300000,0.058771,0.171017,0.155933,0.187237,True,True,delta_t_days < 0
split_30d,30-day sub-threshold order splitting,830,300,314,300,13795099.790000,11495916.490000,12786126.590000,1008973.210000,0.074587,0.135014,0.121433,0.149854,True,True,30-day sub-threshold windows >= T; includes 638 same-day contracts in 241 schools
regime_mismatch,"Regime 1 award in [30k, 50k) BGN excl. VAT",19,17,,17,457652.930000,381377.440000,403706.020000,53946.900000,0.001707,0.007651,0.004782,0.012219,True,False,Statutory threshold in force <= 2023-12-31
regime_mismatch_incl_50k,"Regime 1 awards >= 30,000 BGN excl. VAT (total)",24,22,,22,636582.340000,530485.290000,583268.910000,53313.440000,0.002157,0.009901,0.006548,0.014946,False,,"Includes 5 Regime 1 contracts >= 50,000 BGN excl. VAT"


Date anomalies: 8 (3 pre-2023, 5 post-2026-09-10).
Excluding anomalies: predated = 651 contracts in 379 schools; split_30d = 830 contracts in 300 schools.
Comparative SplittingDetector (incl. above-T co-contracts in 30d trailing window): 1332 contracts in 726 schools.

Top suppliers for predated:


vendor_eik,vendor_name,contracts,schools,value_eur_incl_vat
131468980,",,А1 България“ ЕАД",368,178,"5,185,778.74"
201055457,БАЛИСТИК СЕЛ ЕООД,85,70,"2,227,002.00"
207788103,СКУУЛ КОНСУЛТИНГ ЕООД,47,47,"61,092.43"
203902794,ДЖЕЙ ЕНД ДЖИ ГЛОБАЛ СЪРВИСЕС ЕООД,23,23,"304,014.15"
201463957,ЕНЕРДЖИ СПЕД ЕООД,11,11,"203,431.79"



Top suppliers for split_30d:


vendor_eik,vendor_name,contracts,schools,value_eur_incl_vat
131468980,",,А1 България“ ЕАД",119,47,"1,918,246.40"
206175784,Ера Билд 2020 ЕООД,93,44,"1,747,878.21"
175342714,ЦЕНТЪР ЗА ТВОРЧЕСКО ОБУЧЕНИЕ ООД,75,26,"1,258,830.77"
000885099,Кооперация Панда,68,24,"1,075,670.91"
104518906,ВАЛИ КОМПЮТЪРС ООД,34,12,"547,204.83"



Top suppliers for regime_mismatch:


vendor_eik,vendor_name,contracts,schools,value_eur_incl_vat
201463957,ЕНЕРДЖИ СПЕД ЕООД,5,5,"116,350.60"
207019028,АРБИЛД ГРУП ЕООД,2,2,"50,272.92"
202214115,360 Консулт ЕООД,2,2,"46,231.02"
175063752,КЕЙСИЕМ ЕООД,1,1,"29,482.00"
205101667,ХГ Трейд ЕООД,1,1,"28,677.08"


,part,question,measure,value,interval_low,interval_high,interval_method,decision
0,Section 2,reconciliation,projects in BG-RRP-1.015,2228,,,none,none
1,Section 2,reconciliation,unique beneficiary schools,2227,,,none,none
2,Section 2,reconciliation,projects with contract,2223,,,none,none
3,Section 2,reconciliation,schools with contract,2222,,,none,none
4,Section 2,reconciliation,projects without contract,5,,,none,none
...,...,...,...,...,...,...,...,...
63,Section 4,indicator_split_30d,same_day_sub_split: schools (subset of split_30d),241,,,none,none
64,Section 4,indicator_split_30d,comparative SplittingDetector 30d: contracts,1332,,,none,none
65,Section 4,indicator_split_30d,comparative SplittingDetector 30d: schools,726,,,none,none
66,Section 4,anomalies,anomalous dates total (pre-2023 / post-extract...,8 (3 / 5),,,none,none
